# Oil & Gas Well Pad SCADA — tsqc Quality Control Demo

This notebook runs the full `timeseries-qc` pipeline on one month of synthetic
hourly SCADA data from an oil and gas well pad with three sensor tags:

| Tag | Description | Units | Normal Range |
|-----|-------------|-------|--------------|
| `WHP.PSIG` | Wellhead Pressure | psig | 50-1100 (MAWP) |
| `FMRATE.MSCFD` | Gas flow rate at the meter | Mscfd | 0-5500 |
| `OHT.TEMP_F` | Heater-treater outlet temperature | °F | 50-200 |

**Engineered anomalies** (designed to exercise every built-in rule):

| Anomaly | Tag | Rule triggered | Label |
|---------|-----|----------------|-------|
| Sensor dropout (NaN burst, 20 h) | WHP.PSIG | `NullRule` | bad |
| Wellhead pressure 1500 psig (over MAWP) | WHP.PSIG | `RangeRule` | bad |
| Sub-atmospheric pressure 5 psig | WHP.PSIG | `RangeRule` | bad |
| Comm-freeze flatline 20 h | FMRATE.MSCFD | `FlatlineRule` | sus |
| Flow rate drop -2000 Mscfd | FMRATE.MSCFD | `DeltaRule` | sus |
| Heater-treater stuck 21 h | OHT.TEMP_F | `FlatlineRule` | sus |
| Heater-treater 50 °F drop | OHT.TEMP_F | `DeltaRule` | sus |
| Duplicate timestamp | OHT.TEMP_F | `check_timestamps()` | error |

In [1]:
import sys
sys.path.insert(0, '..')  # allow import from repo root when running in examples/

import pandas as pd
import tsqc

print(f'tsqc version: {tsqc.__version__}')

tsqc version: 0.0.1


## 1. Generate / Load the Synthetic Data

In [2]:
import subprocess, os

csv_path = '../data/oilfield_scada.csv'

if not os.path.exists(csv_path):
    print('Generating synthetic data...')
    subprocess.run([sys.executable, '../data/generate_oilfield_data.py'], check=True)

df = pd.read_csv(csv_path)
print(f'Loaded {len(df):,} rows × {df["tag_name"].nunique()} tags')
print(f'Tags: {sorted(df["tag_name"].unique())}')
print(f'Date range: {df["timestamp"].min()} → {df["timestamp"].max()}')
df.head(8)

Loaded 2,161 rows × 3 tags
Tags: ['FMRATE.MSCFD', 'OHT.TEMP_F', 'WHP.PSIG']
Date range: 2026-04-01T00:00:00+00:00 → 2026-04-30T23:00:00+00:00


,timestamp,tag_name,value
0,2026-04-01T00:00:00+00:00,WHP.PSIG,624.09
1,2026-04-01T01:00:00+00:00,WHP.PSIG,624.66
2,2026-04-01T02:00:00+00:00,WHP.PSIG,631.51
3,2026-04-01T03:00:00+00:00,WHP.PSIG,635.05
4,2026-04-01T04:00:00+00:00,WHP.PSIG,632.94
5,2026-04-01T05:00:00+00:00,WHP.PSIG,646.94
6,2026-04-01T06:00:00+00:00,WHP.PSIG,645.14
7,2026-04-01T07:00:00+00:00,WHP.PSIG,657.03


## 2. Run Quality Checks with YAML Rules

In [3]:
result = tsqc.check(
    df,
    rules='../data/oilfield_rules.yaml',
    assume_tz='UTC',   # CSV has tz-naive ISO strings
)

print(result)
result.df[['timestamp', 'tag_name', 'value', 'quality', 'quality_reasons']].head(12)

QCResult(rows=2161, tags=3)


,timestamp,tag_name,value,quality,quality_reasons
0,2026-04-01 00:00:00+00:00,WHP.PSIG,624.09,good,
1,2026-04-01 01:00:00+00:00,WHP.PSIG,624.66,sus,flatline
2,2026-04-01 02:00:00+00:00,WHP.PSIG,631.51,good,
3,2026-04-01 03:00:00+00:00,WHP.PSIG,635.05,good,
4,2026-04-01 04:00:00+00:00,WHP.PSIG,632.94,good,
5,2026-04-01 05:00:00+00:00,WHP.PSIG,646.94,good,
6,2026-04-01 06:00:00+00:00,WHP.PSIG,645.14,good,
7,2026-04-01 07:00:00+00:00,WHP.PSIG,657.03,good,
8,2026-04-01 08:00:00+00:00,WHP.PSIG,658.16,good,
9,2026-04-01 09:00:00+00:00,WHP.PSIG,665.40,good,


## 3. Quality Summary per Tag

In [4]:
summary = result.summary()
print(summary.to_string(index=False))

    tag_name  total_rows  n_good  n_sus  n_bad  pct_good  pct_sus  pct_bad
    WHP.PSIG         720     691      3     26     95.97     0.42     3.61
FMRATE.MSCFD         720     700     20      0     97.22     2.78     0.00
  OHT.TEMP_F         721     699     22      0     96.95     3.05     0.00


## 4. Interactive Quality Timeline

One horizontal bar per tag. **Red = bad, Yellow = sus, Green = good.**  
Use the **range slider** at the bottom or the **1d / 1w / 1m / All** buttons to zoom.  
Hover any segment to see tag, quality level, start/end time, and duration.

In [5]:
fig = result.plot(
    title='Oil & Gas Well Pad SCADA — April 2026 Quality Timeline',
    height=420,
)
fig.show()

### Zoom: first week with the comm-freeze flatline

In [6]:
# Week 1 (Apr 1-7) contains the FMRATE.MSCFD comm-freeze (rows 60-79)
fig_week = result.plot(
    title='Week 1 Detail — 2026-04-01 to 2026-04-07',
    start='2026-04-01',
    end='2026-04-07T23:00:00+00:00',
)
fig_week.show()

## 5. Timestamp Health Check

In [7]:
ts_issues = result.check_timestamps(expected_freq='1h')
print(f'Timestamp issues found: {len(ts_issues)}')
if not ts_issues.empty:
    print(ts_issues.groupby(['tag_name', 'issue_type', 'severity']).size()
          .reset_index(name='count').to_string(index=False))
ts_issues

Timestamp issues found: 1
  tag_name issue_type severity  count
OHT.TEMP_F  duplicate    error      1


,tag_name,issue_type,timestamp,description,severity
0,OHT.TEMP_F,duplicate,2026-04-26 00:00:00+00:00,Duplicate timestamp: 2026-04-26 00:00:00+00:00,error


## 6. Investigate Individual Tag Issues

In [8]:
# WHP.PSIG — NaN burst, over-range spike, sub-atmospheric excursion
whp = result.df[
    (result.df['tag_name'] == 'WHP.PSIG') &
    (result.df['quality'] != 'good')
].copy()

print(f'WHP.PSIG flagged rows: {len(whp)}')
print(whp['quality_reasons'].value_counts().to_string())
whp[['timestamp', 'value', 'quality', 'quality_reasons']].head(20)

WHP.PSIG flagged rows: 29
quality_reasons
null              20
range|delta        2
delta              2
range              2
range|flatline     2
flatline           1


,timestamp,value,quality,quality_reasons
1,2026-04-01 01:00:00+00:00,624.66,sus,flatline
200,2026-04-09 08:00:00+00:00,NaN,bad,null
201,2026-04-09 09:00:00+00:00,NaN,bad,null
202,2026-04-09 10:00:00+00:00,NaN,bad,null
203,2026-04-09 11:00:00+00:00,NaN,bad,null
204,2026-04-09 12:00:00+00:00,NaN,bad,null
205,2026-04-09 13:00:00+00:00,NaN,bad,null
206,2026-04-09 14:00:00+00:00,NaN,bad,null
207,2026-04-09 15:00:00+00:00,NaN,bad,null
208,2026-04-09 16:00:00+00:00,NaN,bad,null


In [9]:
# FMRATE.MSCFD — comm-freeze flatline and delta spike
fmrate = result.df[
    (result.df['tag_name'] == 'FMRATE.MSCFD') &
    (result.df['quality'] != 'good')
].copy()

print(f'FMRATE.MSCFD flagged rows: {len(fmrate)}')
print(fmrate['quality_reasons'].value_counts().to_string())
fmrate[['timestamp', 'value', 'quality', 'quality_reasons']]

FMRATE.MSCFD flagged rows: 20
quality_reasons
flatline    18
delta        2


,timestamp,value,quality,quality_reasons
782,2026-04-03 14:00:00+00:00,2902.5,sus,flatline
783,2026-04-03 15:00:00+00:00,2902.5,sus,flatline
784,2026-04-03 16:00:00+00:00,2902.5,sus,flatline
785,2026-04-03 17:00:00+00:00,2902.5,sus,flatline
786,2026-04-03 18:00:00+00:00,2902.5,sus,flatline
787,2026-04-03 19:00:00+00:00,2902.5,sus,flatline
788,2026-04-03 20:00:00+00:00,2902.5,sus,flatline
789,2026-04-03 21:00:00+00:00,2902.5,sus,flatline
790,2026-04-03 22:00:00+00:00,2902.5,sus,flatline
791,2026-04-03 23:00:00+00:00,2902.5,sus,flatline


In [10]:
# OHT.TEMP_F — stuck heater and temp drop
oht = result.df[
    (result.df['tag_name'] == 'OHT.TEMP_F') &
    (result.df['quality'] != 'good')
].copy()

print(f'OHT.TEMP_F flagged rows: {len(oht)}')
print(oht['quality_reasons'].value_counts().to_string())

# The stuck heater — 21 consecutive hours
stuck = oht[oht['quality_reasons'].str.contains('flatline', na=False)]
print(f'\nHeater-treater stuck (flatline) period:')
print(f'  Start: {stuck["timestamp"].min()}')
print(f'  End:   {stuck["timestamp"].max()}')
print(f'  Duration: {len(stuck)} hours')

OHT.TEMP_F flagged rows: 22
quality_reasons
flatline    21
delta        1

Heater-treater stuck (flatline) period:
  Start: 2026-04-14 12:00:00+00:00
  End:   2026-04-28 22:00:00+00:00
  Duration: 21 hours


## 7. Export Self-Contained HTML Report

In [11]:
report_path = '../data/oilfield_qc_report.html'
result.export_report(
    report_path,
    title='Oil & Gas Well Pad SCADA — April 2026 Quality Report'
)
import os
print(f'Report written to: {report_path}')
print(f'File size: {os.path.getsize(report_path)/1024:.0f} KB')

Report written to: ../data/oilfield_qc_report.html
File size: 3624 KB


---

## Summary of QC Results

After running `tsqc.check()` with `oilfield_rules.yaml`:

- **WHP.PSIG**: 20 h NaN burst (20 bad rows) + 1500 psig over-MAWP (1 bad row) + 5 psig sub-atmospheric (5 bad rows)
- **FMRATE.MSCFD**: 20 h comm-freeze flatline (sus rows) + -2000 Mscfd delta spike (1 sus row)
- **OHT.TEMP_F**: 21 h heater-stuck flatline (sus rows) + 50 °F drop (1 sus row) + 1 duplicate timestamp

**Key YAML tuning insights:**
- `min_delta: 0.5` on WHP and OHT flatline rules avoids false-positive flags during normal low-volatility operation.
- `min_delta: 5.0` on FMRATE flatline is needed because gas flow rate has wider natural variation than pressure/temperature.
- Range bounds reflect actual equipment limits: MAWP for WHP, physical zero for FMRATE, and heater-treater operating limits for OHT.